In [1]:
from langchain_groq import ChatGroq

In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

## Model:

In [4]:
llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [5]:
print(llm.invoke("What is the capital of France?").content)

<think>

</think>

The capital of France is Paris.


## Embedding Model:

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [7]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [48]:
doc_vector = embedding_model.embed_query("What is the capital of France?")

## 1. DATA INGESTION:

In [9]:
from langchain.document_loaders import PyPDFLoader

In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [11]:
import os

In [12]:
os.getcwd()

'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook'

In [13]:
file_path = os.path.join(os.getcwd(), "data", "LLMAll_en-US_FINAL.pdf")

In [14]:
loader = PyPDFLoader(file_path)

In [15]:
documents = loader.load()

incorrect startxref pointer(1)
parsing for Object Streams


In [17]:
len(documents)

45

In [18]:
# This is an experimental thing. There is no deterministic way to split the text.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len,
)

In [19]:
docs = text_splitter.split_documents(documents)

In [20]:
len(docs)

267

In [21]:
docs[0]

Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1'}, page_content='OWASP PDF v4.2.0a 20241114-202703\nOWASP Top 10 for\nLLM Applications 2025\nVersion 2025\nNovember 18, 2024')

In [22]:
docs[0].metadata

{'producer': 'pypdf',
 'creator': 'PyPDF',
 'creationdate': '',
 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf',
 'total_pages': 45,
 'page': 0,
 'page_label': '1'}

In [23]:
docs[0].page_content

'OWASP PDF v4.2.0a 20241114-202703\nOWASP Top 10 for\nLLM Applications 2025\nVersion 2025\nNovember 18, 2024'

In [24]:
docs[1].page_content

'LICENSE AND USAGE\nThis document is licensed under Creative Commons, CC BY-SA 4.0.\nYou are free to:\n    Share — copy and redistribute the material in any medium or format for any purpose,\n        even commercially.\n    Adapt — remix, transform, and build upon the material for any purpose,\n        even commercially.\n    The licensor cannot revoke these freedoms as long as you follow the license terms.\nUnder the following terms:'

## Store Data in Vector Databse (FAISS):

In [25]:
from langchain.vectorstores import FAISS

In [47]:
doc_vector = embedding_model.embed_documents(docs[0].page_content)

In [27]:
len(embedding_model.embed_documents(docs[0].page_content))

103

In [28]:
vectorstore = FAISS.from_documents(docs, embedding_model)

token->words

chunk--> it is collection of words(token)[characters]

1. in memory(faiss is in memory vector store,chroma)

2. on disk storage(faiss you can persist over the disk,chroma)

3. cloud storage(cloud variant of faiss is not available)(pinecone,weaviate,milvus,mongodbvectorsearch,astradb)

## 2. DATA RETRIEVAL:

From the vectordatabase we are going to fetch or retrive or rank the most appropriate k result

In [29]:
relevant_doc = vectorstore.similarity_search("What is Data Poisoning?")  

In [30]:
relevant_doc

[Document(id='74cb1677-f495-4b40-b02e-b7f9e11738cb', metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf', 'total_pages': 45, 'page': 32, 'page_label': '33'}, page_content='source information, compromising data confidentiality.(Ref #3, #4)\n4. Data Poisoning Attacks\nData poisoning can occur intentionally by malicious actors  (Ref #5, #6, #7) or unintentionally.\nPoisoned data can originate from insiders, prompts, data seeding, or unverified data'),
 Document(id='948b2bee-f57f-424a-aaff-2fed5a5175fe', metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf', 'total_pages': 45, 'page': 19, 'page_label': '20'}, page_content="originate. Data poisoning is considered an integrity attack since tampe

In [31]:
relevant_doc[0].page_content

'source information, compromising data confidentiality.(Ref #3, #4)\n4. Data Poisoning Attacks\nData poisoning can occur intentionally by malicious actors  (Ref #5, #6, #7) or unintentionally.\nPoisoned data can originate from insiders, prompts, data seeding, or unverified data'

In [32]:
relevant_doc[1].page_content

"originate. Data poisoning is considered an integrity attack since tampering with training data\nimpacts the model's ability to make accurate predictions. The risks are particularly high with\nexternal data sources, which may contain unverified or malicious content.\nMoreover, models distributed through shared repositories or open-source platforms can carry\nrisks beyond data poisoning, such as malware embedded through techniques like malicious"

In [33]:
relevant_doc[2].page_content

'Related Frameworks and Taxonomies  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .   15\nLLM04: Data and Model Poisoning .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .   16\nDescription  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .   16'

In [34]:
relevant_doc[3].page_content

'Common risks include degraded model performance, biased or toxic content, and exploitation of\ndownstream systems.\nData poisoning can target different stages of the LLM lifecycle, including pre-training (learning\nfrom general data), fine-tuning (adapting models to specific tasks), and embedding (converting\ntext into numerical vectors). Understanding these stages helps identify where vulnerabilities may\noriginate. Data poisoning is considered an integrity attack since tampering with training data'

In [35]:
relevant_doc = vectorstore.similarity_search("What is Prompt Injection?",k=1)

In [36]:
relevant_doc

[Document(id='bd1496ae-7dbf-4b80-81f8-24b08fca1e8c', metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf', 'total_pages': 45, 'page': 7, 'page_label': '8'}, page_content="methods of prevention for prompt injection. However, the following measures can mitigate the\nimpact of prompt injections:\n1. Constrain model behavior\nProvide specific instructions about the model's role, capabilities, and limitations within the\nsystem prompt. Enforce strict context adherence, limit responses to specific tasks or topics,\nand instruct the model to ignore attempts to modify core instructions.\n2. Define and validate expected output formats")]

In [37]:
relevant_doc[0].page_content

"methods of prevention for prompt injection. However, the following measures can mitigate the\nimpact of prompt injections:\n1. Constrain model behavior\nProvide specific instructions about the model's role, capabilities, and limitations within the\nsystem prompt. Enforce strict context adherence, limit responses to specific tasks or topics,\nand instruct the model to ignore attempts to modify core instructions.\n2. Define and validate expected output formats"

#### you can explore about keyword filtering

In [44]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [45]:
retriever.invoke("How to prevent against Prompt Injection attacks?")

[Document(id='bd1496ae-7dbf-4b80-81f8-24b08fca1e8c', metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Portal\\notebook\\data\\LLMAll_en-US_FINAL.pdf', 'total_pages': 45, 'page': 7, 'page_label': '8'}, page_content="methods of prevention for prompt injection. However, the following measures can mitigate the\nimpact of prompt injections:\n1. Constrain model behavior\nProvide specific instructions about the model's role, capabilities, and limitations within the\nsystem prompt. Enforce strict context adherence, limit responses to specific tasks or topics,\nand instruct the model to ignore attempts to modify core instructions.\n2. Define and validate expected output formats"),
 Document(id='950b40b9-9c8d-45c8-b9f4-51e57ef7a234', metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'c:\\Users\\Support\\Documents\\Soumadeep_Local\\LLMOPs\\Project1_Document_Por

#### Question: user question/query

#### Context: based on the question retrieving the info from the vector database

## PROMPT TEAMPLATE:

In [38]:
prompt_template = """
        Answer the question based on the context provided below. 
        If the context does not contain sufficient information, respond with: 
        "I do not have enough information about this."

        Context: {context}

        Question: {question}

        Answer:"""

In [39]:
from langchain.prompts import PromptTemplate

In [40]:
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [41]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n        Answer the question based on the context provided below. \n        If the context does not contain sufficient information, respond with: \n        "I do not have enough information about this."\n\n        Context: {context}\n\n        Question: {question}\n\n        Answer:')

## CHAINING:

In [42]:
from langchain_core.output_parsers import StrOutputParser

In [43]:
parser = StrOutputParser()

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [46]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("How to prevent against supply chain attacks?")